# Notebook 04 - Prosodic Features

## Goal
Extract physiology-related features like pitch and energy contours.


## Agenda
- Estimate F0 contour
- Compute RMS energy
- Derive jitter and shimmer proxies
- Estimate HNR-like and speaking-rate proxies


## Concept and Math

Prosody describes how speech is produced over time (pitch, intensity, rhythm).
Jitter and shimmer quantify micro-variability in pitch period and amplitude.
Synthetic speech often appears over-smoothed in these dimensions.


In [ ]:
from pathlib import Path
import numpy as np
import librosa as lb
import librosa.display
import matplotlib.pyplot as plt

DATA_ROOT = Path("../dataset")
audio_files = sorted(DATA_ROOT.rglob("*.flac")) + sorted(DATA_ROOT.rglob("*.wav"))
if not audio_files:
    raise FileNotFoundError("No .flac or .wav found under ../dataset")

audio_path = audio_files[0]
print(f"Using: {audio_path}")

wave, sr = lb.load(audio_path, sr=16000, mono=True)

f0, _, _ = lb.pyin(wave, fmin=lb.note_to_hz("C2"), fmax=lb.note_to_hz("C7"), sr=sr, hop_length=160)
rms = lb.feature.rms(y=wave, hop_length=160)[0]

f0v = f0[~np.isnan(f0)]
jitter_proxy = np.mean(np.abs(np.diff(f0v)) / (f0v[:-1] + 1e-8)) if len(f0v) > 2 else np.nan
shimmer_proxy = np.mean(np.abs(np.diff(rms)) / (rms[:-1] + 1e-8)) if len(rms) > 2 else np.nan

harm, perc = lb.effects.hpss(wave)
hnr_like_db = 10 * np.log10((np.sum(harm ** 2) + 1e-8) / (np.sum(perc ** 2) + 1e-8))

print("jitter_proxy:", jitter_proxy)
print("shimmer_proxy:", shimmer_proxy)
print("hnr_like_db:", hnr_like_db)


## PyTorch Equivalent Snippet
Understand the librosa block first, then map it to this snippet.


In [ ]:
import torchaudio

wave_t, sr_t = torchaudio.load(str(audio_path))
wave_t = wave_t.mean(dim=0, keepdim=True)
pitch_t = torchaudio.functional.detect_pitch_frequency(wave_t, sr_t)
print("pitch_t shape:", pitch_t.shape)


## Review Checklist
- Which values are exact vs proxy measures?
- Why are prosodic cues useful for deepfake detection?
- How does voiced/unvoiced handling affect F0 statistics?
